# [M7] DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
## Buổi S03: Thử Nghiệm Mô Hình Nâng Cao & Ablation Study

---

### Metadata Khóa Học
* **Module:** M7 — DỰ ÁN AI THẬT ĐỂ ĐƯA VÀO PORTFOLIO
* **Buổi:** S03
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook thực nghiệm Ablation Study và bảng theo dõi mô hình chuẩn mực
* **Mức độ liên quan VOAI / Olympic AI:** Chứng minh giá trị khoa học định lượng của giải pháp trước hội đồng giám khảo.

#### Mục tiêu học tập (Learning Objectives)
- Đưa các kiến trúc mô hình nâng cao vào thực nghiệm: Rừng ngẫu nhiên (Random Forest) và Mạng nơ-ron sâu (PyTorch MLP).
- Thực hiện Tinh chỉnh Siêu tham số có hệ thống (Systematic Hyperparameter Tuning) bằng Random Search hoặc Grid Search.
- Tiến hành Nghiên cứu loại trừ (Ablation Study) để chứng minh một cách khoa học đóng góp định lượng của từng thành phần trong mô hình.
- Xây dựng Bảng Theo dõi Thực nghiệm (Experiment Tracking Table) ghi nhận chi tiết lịch sử thử nghiệm đạt chuẩn nghiên cứu.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M7_S02 (Đã có Preprocessing Pipeline và mốc Baseline).
- Nắm vững kỹ năng huấn luyện PyTorch và Sklearn từ các module trước.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Trong giới nghiên cứu và phát triển AI chuyên nghiệp, khi bạn tuyên bố: *"Mô hình của tôi đạt độ chính xác 92%"*, câu hỏi chất vấn đầu tiên của các chuyên gia phản biện sẽ là: **"Nhờ đâu mà bạn đạt được con số đó? Do bạn thêm đặc trưng mới, do kiến trúc mạng sâu hơn, hay đơn giản chỉ là may mắn ngẫu nhiên?"**.

Nếu bạn không thể trả lời câu hỏi này, giải pháp của bạn sẽ bị coi là thiếu cơ sở khoa học. **Nghiên cứu loại trừ (Ablation Study)** là phương pháp tiêu chuẩn vàng của các bài báo khoa học quốc tế đỉnh cao: Bạn lần lượt tháo bỏ từng thành phần (bỏ feature scaling, bỏ dropout, bỏ đặc trưng tương tác) để đo lường chính xác "giá trị gia tăng" của từng bộ phận. Buổi học hôm nay biến dự án của bạn thành một công trình nghiên cứu thực thụ!



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Phương pháp Nghiên cứu Loại trừ (Ablation Study)
* **Khái niệm:** Là chuỗi thực nghiệm có kiểm soát nhằm cô lập và định lượng tác động của một thành phần cụ thể trong hệ thống AI.
* **Quy trình chuẩn:**
  1. Mô hình Đầy đủ (Full Model): Có tất cả các đặc trưng và kỹ thuật.
  2. Thử nghiệm A: Full Model **loại bỏ** Feature Engineering.
  3. Thử nghiệm B: Full Model **loại bỏ** Dropout / Regularization.
  4. Thử nghiệm C: Full Model **thay thế** bộ tối ưu hóa Adam bằng SGD.
* Kết quả được tổng hợp thành bảng để chứng minh tính cần thiết của từng khối kỹ thuật!

### 2. Tinh chỉnh Siêu tham số (Hyperparameter Optimization)
* Không tinh chỉnh mò mẫm bằng tay!
* Sử dụng `RandomizedSearchCV` hoặc `GridSearchCV`:
  - Khám phá không gian siêu tham số: `n_estimators`, `max_depth`, `learning_rate`, `weight_decay`.
  - Luôn đo lường trên sơ đồ K-Fold Cross-Validation để tránh Overfitting trên tập kiểm thử.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Bản đồ Nghiên cứu Loại trừ (Ablation Study Workflow):
[MÔ HÌNH ĐẦY ĐỦ: Deep MLP + Features + Dropout] ──────────► [F1: 0.885] (Đỉnh cao nhất)
                 │
                 ├── [Loại bỏ Dropout] ───────────────────► [F1: 0.842] (Mất -4.3% do Overfit)
                 ├── [Loại bỏ Feature Engineering] ───────► [F1: 0.810] (Mất -7.5% do thiếu tín hiệu)
                 └── [Thay Deep MLP bằng Baseline] ───────► [F1: 0.745] (Mất -14.0%)
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. KHOI TAO BANG THEO DOI THUC NGHIEP (EXPERIMENT TRACKING TABLE)
# Gia lap tap du lieu dac trung phong phu (N=500, 8 dac trung)
np.random.seed(42)
N = 500
X_raw = np.random.randn(N, 5)
# Tao 3 dac trung tuong tac bo sung tu Feature Engineering
feat_interaction_1 = (X_raw[:, 0] * X_raw[:, 1]).reshape(-1, 1)
feat_ratio_2 = (X_raw[:, 2] / (np.abs(X_raw[:, 3]) + 0.1)).reshape(-1, 1)
feat_poly_3 = (X_raw[:, 4] ** 2).reshape(-1, 1)

X_full = np.hstack([X_raw, feat_interaction_1, feat_ratio_2, feat_poly_3])
y = (X_full[:, 0] * 1.5 + X_full[:, 5] * 2.0 - X_full[:, 6] + np.random.randn(N) > 0).astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Tap du lieu nghien cuu: {X_full.shape[0]} mau, {X_full.shape[1]} dac trung tong cong.")

### 🔍 Code Walkthrough (Khối 1)
Khối code thiết lập môi trường thực nghiệm có đối chứng:
- Tách bạch rõ ràng giữa 5 đặc trưng gốc (`X_raw`) và 3 đặc trưng tương tác sinh ra từ Feature Engineering.
- Sẵn sàng cho chuỗi thực nghiệm Ablation Study bóc tách đóng góp của từng nhóm.


In [ ]:
# 2. TIEN HANH ABLATION STUDY & BENCHMARK CAC MO HINH
ablation_experiments = []

# Thuc nghiem 1: Baseline Co ban (Logistic Regression tren dac trung goc)
scores_exp1 = cross_val_score(LogisticRegression(), X_raw, y, cv=skf, scoring='f1_macro')
ablation_experiments.append({
    'Experiment': '1. Baseline (Logistic Reg + Raw Features)',
    'Features Used': '5 Raw Features',
    'Model Type': 'Linear Model',
    'CV Macro-F1': f"{scores_exp1.mean():.4f} (+/- {scores_exp1.std():.3f})",
    'Delta vs Baseline': '0.00% (Moc chuan)'
})

# Thuc nghiem 2: Random Forest tren dac trung goc (Kiem tra suc manh phi tuyen)
rf_raw = RandomForestClassifier(n_estimators=80, max_depth=4, random_state=42)
scores_exp2 = cross_val_score(rf_raw, X_raw, y, cv=skf, scoring='f1_macro')
ablation_experiments.append({
    'Experiment': '2. Non-linear Tree (RF + Raw Features)',
    'Features Used': '5 Raw Features',
    'Model Type': 'Random Forest',
    'CV Macro-F1': f"{scores_exp2.mean():.4f} (+/- {scores_exp2.std():.3f})",
    'Delta vs Baseline': f"+{(scores_exp2.mean() - scores_exp1.mean())*100:.2f}%"
})

# Thuc nghiem 3: FULL MODEL (Random Forest + Full Engineered Features)
rf_full = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
scores_exp3 = cross_val_score(rf_full, X_full, y, cv=skf, scoring='f1_macro')
ablation_experiments.append({
    'Experiment': '3. FULL SOLUTION (RF + Feature Engineering)',
    'Features Used': '8 Full Features',
    'Model Type': 'Tuned Random Forest',
    'CV Macro-F1': f"{scores_exp3.mean():.4f} (+/- {scores_exp3.std():.3f})",
    'Delta vs Baseline': f"+{(scores_exp3.mean() - scores_exp1.mean())*100:.2f}%"
})

df_ablation = pd.DataFrame(ablation_experiments)
print("===== BANG KET QUA NGHIEN CUU LOAI TRU (ABLATION STUDY REPORT) =====")
print(df_ablation.to_string(index=False))
print("\n-> KET LUAN KHOA HOC: Feature Engineering dong gop +{:.2f}% F1, trong khi thuat toan phi tuyen dong gop +{:.2f}% F1!".format(
    (scores_exp3.mean() - scores_exp2.mean())*100, (scores_exp2.mean() - scores_exp1.mean())*100
))

### 🔍 Code Walkthrough (Khối 2)
Khối code thực thi chuỗi thực nghiệm Ablation Study đạt chuẩn nghiên cứu khoa học:
- Đo lường phương sai ổn định qua 5-Fold Cross Validation (`mean +/- std`).
- Bóc tách rạch ròi bằng số liệu: Thuật toán phi tuyến đóng góp bao nhiêu phần trăm, và Kỹ nghệ đặc trưng đóng góp bao nhiêu phần trăm vào chiến thắng chung cuộc.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Thử nghiệm với thuật toán Gradient Boosting
Import `HistGradientBoostingClassifier` từ `sklearn.ensemble`. Thêm thực nghiệm số 4 vào bảng theo dõi và quan sát xem GBDT có vượt qua Random Forest hay không.

### Bài 2 (Easy): Tinh chỉnh siêu tham số bằng RandomizedSearchCV
Sử dụng `sklearn.model_selection.RandomizedSearchCV` để tìm bộ siêu tham số tối ưu cho Random Forest với không gian tìm kiếm: `n_estimators: [50, 100, 150]`, `max_depth: [3, 5, 7]`. In ra `best_params_`.

### Bài 3 (Medium): Đo lường chi phí tính toán (Computational Cost Tracking)
Thêm cột `Inference Time (ms)` vào bảng theo dõi thực nghiệm. Viết đoạn code đo thời gian thực hiện dự đoán cho 1000 mẫu của từng mô hình và thảo luận về sự đánh đổi giữa độ chính xác và độ trễ.

### Bài 4 (Medium): Bóc tách tác động của từng đặc trưng (Individual Feature Ablation)
Lần lượt loại bỏ từng đặc trưng một trong 3 đặc trưng tương tác và đo lường sự sụt giảm F1-Score. Cho biết đặc trưng nào trong 3 đặc trưng đó đóng vai trò "gánh team" quan trọng nhất.

### Bài 5 (Challenge): Xuất báo cáo Ablation sang định dạng Bảng LaTeX / Markdown
Viết một hàm Python `export_ablation_latex(df_ablation, filename="ablation_table.tex")` tự động chuyển đổi bảng kết quả thành mã nguồn bảng LaTeX chuẩn của các hội nghị AI quốc tế (như NeurIPS hoặc CVPR).



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Một trong những tiêu chí chấm điểm khắt khe nhất của Ban giám khảo Olympic AI và các chuyên gia hội đồng thẩm định là **Tính Thuyết Phục Kỹ Thuật (Technical Justification)**:
Một bài thuyết trình chỉ nói: *"Em thử linh tinh và thấy mô hình này điểm cao nhất"* sẽ bị đánh giá rất thấp. Ngược lại, thí sinh trình bày một bảng **Ablation Study** rõ ràng, chỉ ra chính xác quyết định kỹ thuật nào mang lại bao nhiêu phần trăm cải thiện điểm số, sẽ ngay lập tức chinh phục hoàn toàn hội đồng và khẳng định đẳng cấp vượt trội.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh tiến hành Ablation Study nhưng mỗi lần chạy lại đổi `random_state` khác nhau hoặc đổi kích thước fold, khiến sự chênh lệch điểm số thực chất chỉ do nhiễu ngẫu nhiên chứ không phải do mô hình:

```python
# CODE CÓ LỖI THỰC NGHIỆM KHÔNG CÙNG MẶT BẰNG ĐỐI CHỨNG:
# Exp 1: cv = StratifiedKFold(n_splits=5, random_state=42)
# Exp 2: cv = StratifiedKFold(n_splits=5, random_state=999) # Doi seed -> Sai lech mat bang so sanh!
```

**Nhiệm vụ của bạn:**
1. Khẳng định nguyên tắc thực nghiệm khoa học: **Mọi thí nghiệm trong Ablation Study bắt buộc phải chia sẻ CHUNG MỘT SƠ ĐỒ FOLD DUY NHẤT** để đảm bảo sự so sánh là tuyệt đối công bằng.
2. Cố định đối tượng `skf` làm chuẩn quy chiếu chung cho tất cả các cuộc so găng.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Mục tiêu tối thượng của phương pháp "Ablation Study" (Nghiên cứu loại trừ) trong AI là gì?
   - A. Xóa toàn bộ mã nguồn của dự án.
   - B. Lần lượt cô lập và tháo bỏ từng thành phần kỹ thuật để đo lường định lượng đóng góp thực tế của từng bộ phận, chứng minh tính khoa học của giải pháp.
   - C. Tự động dịch chuyển dữ liệu sang máy chủ khác.
   - D. Giảm kích thước file nộp bài.

2. Tại sao việc tinh chỉnh siêu tham số (Hyperparameter Tuning) luôn phải được đánh giá qua Cross-Validation thay vì chỉ đo trên một tập kiểm tra duy nhất?
   - A. Vì Cross-Validation chạy nhanh hơn.
   - B. Để ngăn ngừa hiện tượng mô hình bị Overfitting gián tiếp lên tập kiểm tra đó (Tuning Leakage) và bảo đảm tính tổng quát hóa ổn định trên nhiều phân vùng dữ liệu khác nhau.
   - C. Vì luật thi đấu bắt buộc.
   - D. Do Scikit-Learn yêu cầu.

3. Điểm khác biệt cơ bản giữa `GridSearchCV` và `RandomizedSearchCV` là gì?
   - A. `GridSearchCV` duyệt qua tất cả các tổ hợp có thể (chậm nhưng vét cạn), trong khi `RandomizedSearchCV` lấy mẫu ngẫu nhiên một số tổ hợp (nhanh hơn rất nhiều và hiệu quả trong không gian tham số lớn).
   - B. `GridSearchCV` chỉ dùng được cho Deep Learning.
   - C. `RandomizedSearchCV` luôn cho kết quả kém hơn.
   - D. Không có sự khác biệt nào.

4. Trong bảng theo dõi thực nghiệm (Experiment Tracking Table), giá trị độ lệch chuẩn `+/- std` bên cạnh điểm trung bình F1 biểu thị điều gì?
   - A. Số lượng tham số của mô hình.
   - B. Mức độ ổn định và độ biến thiên của mô hình qua các fold dữ liệu khác nhau: Độ lệch chuẩn càng nhỏ thì mô hình càng đáng tin cậy.
   - C. Thời gian chạy tính bằng phút.
   - D. Số lượng epochs đã hoàn thành.

5. Một giải pháp AI đạt chuẩn nghiên cứu khoa học bắt buộc phải chứng minh được điều gì trước hội đồng thẩm định?
   - A. Đã sử dụng máy tính đắt tiền nhất.
   - B. Điểm số cải tiến vượt bậc so với Baseline và có số liệu thực nghiệm minh chứng cho đóng góp của từng thành phần kỹ thuật.
   - C. Đã viết hơn 10,000 dòng code.
   - D. Đã dùng tất cả các thư viện có sẵn trên mạng.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Bản chất và phương pháp luận của Nghiên cứu Loại trừ (Ablation Study).
  - Kỹ thuật tinh chỉnh siêu tham số có hệ thống (Hyperparameter Optimization).
  - Cách thiết lập Bảng Theo dõi Thực nghiệm (Experiment Tracking Table) chuẩn quốc tế.
* **What I can now do:**
  - Chứng minh một cách khoa học giá trị của từng kỹ thuật trong hệ thống AI của mình.
  - Tự tin bảo vệ giải pháp kỹ thuật trước mọi hội đồng phản biện khó tính nhất.
* **What I still need to practice:**
  - Phân tích lỗi sai sâu sắc và xây dựng **Giao diện Demo Tương tác (Interactive Demo)** ở Buổi 4!



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M7_S04**, chúng ta sẽ biến mô hình thành một sản phẩm có thể tương tác: **Đánh Giá Chi Tiết, Phân Tích Lỗi Sai (Error Analysis) & Xây Dựng Interactive Demo** — Khám phá các ca thất bại nặng nề nhất, đánh giá hiệu năng trên từng phân nhóm và tự tay lập trình ứng dụng Demo tương tác trực tiếp!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M7**:
- Thực nghiệm mô hình nâng cao & Tinh chỉnh siêu tham số ✓
- Nghiên cứu loại trừ (Ablation Study) bóc tách đóng góp kỹ thuật ✓
- Bảng theo dõi thực nghiệm (Experiment tracking table) chuẩn khoa học ✓

* **Session Deliverable:** Mã nguồn Ablation Study Framework + Bảng theo dõi thực nghiệm có đối chứng
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
